# 05 · Objects, classes, and exceptions

A class gives a name to a kind of object and groups its data with operations on that data.
In this chapter you will build classes, explain how methods receive `self`, distinguish shared
class attributes from each object's state, and use Python's operators and exception system.

**Before you begin:** you should be comfortable with functions, lists, dictionaries, and loops.
Run cells from top to bottom. All examples are independent of other notebooks and use only
Python's standard library. Expected errors are caught so **Restart Kernel and Run All Cells** works.
The six exercise cells are safe starting points; their solutions are in
[05_oop_solutions.ipynb](../solutions/05_oop_solutions.ipynb).


## 1. Choose a useful way to organize a program

Python supports several programming styles. Procedural code expresses a sequence of steps;
object-oriented code groups state with methods; functional code composes functions that
transform values. Declarative code describes a desired result. These styles can be combined.

A brief functional bridge: `map` applies a function, `filter` keeps matching values, and a
`lambda` creates a small function expression. In Python 3, `map` and `filter` return lazy
iterators, so `list` below requests all the finite results. Functions can also receive and
return other functions; that is the basis of decorators. Chapter 06 develops these ideas.
Here we focus on objects whose state changes in a controlled way.


In [ ]:
languages = ["python", "perl", "java", "c++"]
print(list(map(len, languages)))
print(list(filter(lambda length: length > 3, map(len, languages))))


## 2. Objects, names, and namespaces

Every Python object has a type and an identity. A name refers to an object; assigning another
name does not copy it. A **namespace** maps names to objects, such as a module's namespace.
An **attribute** is reached with a dot, as in `math.pi` or `point.x`.

Use `is` when identity is the question and `==` when values are the question. The numeric
value from `id` is useful for comparing identity during an object's lifetime; it is not a
permanent identifier to save. Here two names refer to one list.


In [ ]:
import math

measurements = [3, 4]
same_measurements = measurements
same_measurements.append(5)
print(measurements)
print(same_measurements is measurements)
print(type(measurements).__name__, math.pi)


## 3. Defining a class creates a class object

The `class` statement runs its body in a new namespace and binds the resulting class object
to a name. The body usually contains assignments and function definitions. It has not yet
created an instance of that class. `ClassName.attribute` reads a class attribute, and
`ClassName()` normally creates and initializes an instance.

Classes are themselves objects: `type(Greeter)` is `type`. An instance has its class as its
type. The class's `__dict__` exposes a read-only mapping view of its namespace; update class
attributes through normal assignment rather than editing that mapping.


In [ ]:
class Greeter:
    """A small type with shared data and behavior."""
    greeting = "Hello"

    def greet(self):
        return self.greeting

print(type(Greeter).__name__)
print(Greeter.greeting, Greeter.__dict__["greeting"])
visitor = Greeter()
print(type(visitor).__name__, visitor.greet())


## 4. Class objects and instance objects play different roles

The class describes the behavior shared by its instances. Each call below makes a distinct
`Greeter`, even though both initially return the same greeting. An attribute assigned on one
instance can specialize that instance. An instance can also support operators, indexing,
iteration, or even calling when its class implements the corresponding special methods.

You have already called classes: `int("42")`, `list("abc")`, and `dict(a=1)` construct built-in
objects. `isinstance(value, SomeClass)` also recognizes instances of subclasses.


In [ ]:
first = Greeter()
second = Greeter()
first.greeting = "Welcome"
print(first.greet(), second.greet(), Greeter.greeting)
print(first is second, isinstance(first, Greeter))
print(int("101001", base=2), list("hap.py"), dict(a=1, b=2))


## 5. `__init__` initializes an already created instance

When a class is called, Python normally creates an instance with `__new__`, then calls
`__init__` to initialize it. You usually only need `__init__`. It receives the new instance as
its first argument, conventionally named `self`, and must return `None` (normally implicitly).
It does **not** create the class, and is more precisely an initializer than a constructor.

`self.real` belongs to the instance; `real` is the parameter local to this method call.
Default and keyword arguments offer flexible initialization. Defining `__init__` a second
time would replace the first definition; Python does not select between method definitions
by argument types or signatures. A named factory, often a `@classmethod`, is another option.


In [ ]:
class ComplexValue:
    def __init__(self, real=0, imaginary=0):
        self.real = real
        self.imaginary = imaginary

c = ComplexValue(3.0, imaginary=-4.5)
origin = ComplexValue()
print(c.real, c.imaginary)
print(origin.real, origin.imaginary)


## 6. Instance attributes and lookup

For the ordinary data attributes in these examples, Python checks the instance and then
the class and its base classes. An instance attribute can **shadow** a class attribute;
deleting it reveals the class value again. Properties and other descriptors add rules to
this simplified model, so it is not a universal description of all attribute lookup.

Ordinary user-defined instances store attributes in `__dict__`; `vars(instance)` lets us
inspect that dictionary. Some types use other storage, such as `__slots__`, and have no
instance dictionary. New attributes can be assigned and deleted, but a predictable set of
attributes initialized in `__init__` usually makes a class easier to understand.


In [ ]:
class Score:
    value = 100

    def __init__(self):
        self.value = 0

score = Score()
print(score.value, vars(score))
del score.value
print(score.value, vars(score))
score.note = "practice"
print(score.note)
del score.note


## 7. A bound method pairs a function with an instance

A function defined in a class becomes a **bound method** when accessed through an instance.
Calling `visitor.greet()` supplies `visitor` automatically as `self`. In this ordinary case,
it is equivalent to `Greeter.greet(visitor)`. `self` is a naming convention, not a keyword,
but following it makes code recognizable to other Python programmers.

A bound method remembers both its object (`__self__`) and its function (`__func__`). You can
store the method in a variable and call it later. Class methods instead receive `cls`, and
static methods receive no automatic first argument; those are different method kinds.


In [ ]:
saved_greeting = visitor.greet
print(type(Greeter.greet).__name__, type(saved_greeting).__name__)
print(saved_greeting.__self__ is visitor)
print(saved_greeting.__func__ is Greeter.greet)
print(saved_greeting(), Greeter.greet(visitor))


## 8. A class with state: pizza slices

An object is useful when several operations work with the same state. A `Pizza` stores its
radius, toppings, and remaining slices. `eat_slice` changes only that pizza and reports
whether a slice was available. A return value lets its caller decide what to display.

Initialization rejects a nonpositive radius and invalid slice counts. `bool` is a subclass
of `int`, so the explicit boolean check prevents accepting `True` as a slice count. Toppings
are copied to an immutable tuple. `__repr__` gives a useful representation for debugging.


In [ ]:
class Pizza:
    """Track the remaining slices of one pizza."""

    def __init__(self, radius, toppings=(), slices=8):
        if radius <= 0:
            raise ValueError("radius must be positive")
        if isinstance(slices, bool) or not isinstance(slices, int):
            raise TypeError("slices must be an integer")
        if slices < 0:
            raise ValueError("slices must not be negative")
        self.radius = radius
        self.toppings = tuple(toppings)
        self.slices_left = slices

    def eat_slice(self):
        """Consume a slice and return True, or return False when empty."""
        if self.slices_left == 0:
            return False
        self.slices_left -= 1
        return True

    def __repr__(self):
        return (f"Pizza(radius={self.radius!r}, toppings={self.toppings!r}, "
                f"slices={self.slices_left!r})")

pizza = Pizza(14, ("Pepperoni", "Olives"), slices=2)
print(repr(pizza))
print(pizza.eat_slice(), pizza.eat_slice(), pizza.eat_slice())
print(pizza.slices_left)


## 9. Class state and instance state

A class attribute such as `kind` supplies a shared default. An instance attribute such as
`name` describes one dog. Reassigning a class attribute changes what unshadowed instances
see; assigning `dog.kind` creates an instance attribute instead of changing every dog.

Use a class attribute for intentionally shared information, such as a category. Use an
instance attribute for data that belongs to one object. Names written in `UPPER_CASE` are
conventional constants, but Python does not enforce that convention.


In [ ]:
class NamedDog:
    kind = "Canine"

    def __init__(self, name):
        self.name = name

astro = NamedDog("Astro")
buddy = NamedDog("Buddy")
print(astro.kind, buddy.kind, astro.name, buddy.name)
astro.kind = "Astronaut canine"
print(astro.kind, buddy.kind, NamedDog.kind)


## 10. Two shared mutable state traps

The first class below has one list on the **class**, so every instance finds and mutates the
same list. Appending does not assign a new instance attribute. The second class has a list
as a **default argument**. That list is created once when the function definition runs and
reused whenever the argument is omitted. Moving `[]` into a default argument does not fix
the bug. The different mechanisms produce the same surprising sharing.


In [ ]:
class SharedDog:
    tricks = []

    def teach_trick(self, trick):
        self.tricks.append(trick)

one, two = SharedDog(), SharedDog()
one.teach_trick("sit")
print(two.tricks, one.tricks is two.tricks)

class DefaultDog:
    def __init__(self, tricks=[]):  # Deliberate bug for this demonstration.
        self.tricks = tricks

three, four = DefaultDog(), DefaultDog()
three.tricks.append("roll over")
print(four.tricks, three.tricks is four.tricks)


## 11. Give each instance its own collection

Create a new list inside `__init__`. If callers may supply initial values, use `None` as the
default and copy the supplied iterable. This also prevents later changes to the caller's
list from changing the dog's list unexpectedly. `list(tricks)` is a shallow copy: nested
mutable elements would still be shared. Here every trick is an immutable string.


In [ ]:
class Dog:
    """Each dog owns its own list of tricks, including when a list is supplied."""

    kind = "Canine"

    def __init__(self, name, tricks=None):
        self.name = name
        self.tricks = [] if tricks is None else list(tricks)

    def teach_trick(self, trick):
        self.tricks.append(trick)

starting_tricks = ["sit"]
fido = Dog("Fido", starting_tricks)
buddy = Dog("Buddy")
starting_tricks.append("stay")
fido.teach_trick("roll over")
print(fido.tricks, buddy.tricks, starting_tricks)
assert fido.tricks is not buddy.tricks


## 12. Privacy and readable public interfaces

Use verbs for methods (`teach_trick`) and nouns for data (`name`). A leading underscore
(`_internal`) marks a non-public implementation detail by convention. A double leading
underscore (`__name`) triggers **name mangling**, mainly to reduce accidental collisions
with subclass attributes. It is not a security boundary. Names such as `__repr__` with both
leading and trailing double underscores are reserved language hooks, not private names.

A property can expose a computed or read-only public attribute while keeping the storage
detail separate. This property has no setter, so assigning `label.name` raises `AttributeError`.
The convention still asks callers to leave `_name` alone.


In [ ]:
class Label:
    def __init__(self, name):
        self._name = name
        self.__format_version = 1

    @property
    def name(self):
        return self._name

label = Label("Python")
print(label.name, sorted(vars(label)))
try:
    label.name = "Changed"
except AttributeError:
    print("The public name property is read-only.")


## 13. Inheritance and `super`

`class OnlineCourse(Course)` makes a subclass. It inherits behavior and may override a
method. An override replaces the method found through ordinary lookup; it can use `super()`
to extend the inherited behavior. Call `super().__init__` if the base initializer has work
that must still happen. All ordinary Python 3 classes ultimately inherit from `object`.

`isinstance` recognizes an instance through its base classes, and `issubclass` compares
classes. Use inheritance when a subclass can sensibly stand in for the base class. Simply
storing a helper object as an attribute is often enough when there is no such relationship.


In [ ]:
class Course:
    """A base class whose method a subclass can extend."""

    def __init__(self, title):
        self.title = title

    def describe(self):
        return self.title


class OnlineCourse(Course):
    def __init__(self, title, platform):
        super().__init__(title)
        self.platform = platform

    def describe(self):
        return f"{super().describe()} on {self.platform}"

course = OnlineCourse("Python", "Jupyter")
print(course.describe())
print(isinstance(course, Course), issubclass(OnlineCourse, Course))
print([base.__name__ for base in OnlineCourse.mro()])


## 14. Multiple inheritance uses C3 method resolution

With several base classes, Python computes a **method resolution order** (MRO), visible as
`Class.mro()` or `Class.__mro__`. The C3 algorithm respects declared base order and preserves
the ordering established by parent classes. It is **not breadth-first search**. If the
constraints conflict, Python rejects the class definition with `TypeError`.

`super()` continues after the current class in the actual instance's MRO; it does not
simply mean "call my first parent." In this diamond, cooperative `describe` methods each
call `super` once and have compatible signatures, so the shared base is reached once.
The second example reproduces the more complex hierarchy from the source material.


In [ ]:
class Root:
    def describe(self):
        return ["Root"]

class Left(Root):
    def describe(self):
        return ["Left"] + super().describe()

class Right(Root):
    def describe(self):
        return ["Right"] + super().describe()

class Diamond(Left, Right):
    pass

print([base.__name__ for base in Diamond.mro()])
print(Diamond().describe())

class A: pass
class B: pass
class C: pass
class D: pass
class E: pass
class K1(A, B, C): pass
class K2(D, B, E): pass
class K3(D, A): pass
class Z(K1, K2, K3): pass

print([base.__name__ for base in Z.mro()])
assert [base.__name__ for base in Z.mro()] == [
    "Z", "K1", "K2", "K3", "D", "A", "B", "C", "E", "object"
]


## 15. Special methods let objects follow Python protocols

A protocol is a set of operations an object supports. Implement the relevant **dunder**
(double underscore) methods and ordinary Python syntax can use your object. Prefer the
ordinary syntax when using an object: write `len(shelf)`, not `shelf.__len__()`.

| Operation | Main special method |
|---|---|
| `str(x)` / `repr(x)` | `__str__` / `__repr__` |
| `x + y`, `x == y`, `x < y` | `__add__`, `__eq__`, `__lt__` |
| `len(x)`, `item in x`, `x[index]` | `__len__`, `__contains__`, `__getitem__` |
| `iter(x)`, `next(iterator)` | `__iter__`, `__next__` |

These are the main hooks, not exact rewrites: operators can use reflected methods, fallback
rules, and subclass priority. A container can return a separate iterator instead of being
an iterator itself. Here `ReadingList` delegates to a normal list, so it can be iterated repeatedly.


In [ ]:
class ReadingList:
    """A small container that delegates its operations to a private list."""

    def __init__(self, titles=()):
        self._titles = list(titles)

    def __len__(self):
        return len(self._titles)

    def __contains__(self, title):
        return title in self._titles

    def __getitem__(self, index):
        return self._titles[index]

    def __iter__(self):
        return iter(self._titles)

shelf = ReadingList(["Python", "Data", "Design"])
print(len(shelf), "Data" in shelf, shelf[0], shelf[-1])
print(shelf[1:], list(shelf), list(shelf))


## 16. A Point that works with operators

`rotate_90_ccw` mutates a point: `(x, y)` becomes `(-y, x)`. Addition returns a **new** point.
`__eq__` compares coordinates; for an unsupported type, return the special value
`NotImplemented` so Python can try the other operand's implementation or its fallback.
`NotImplemented` is a value, distinct from the exception class `NotImplementedError`.

`__repr__` aims to be precise and useful to a programmer; `__str__` is the friendly display.
`sum` normally starts with integer zero. Supporting `point + point` does not automatically
support `0 + point`, so supply `start=Point()` explicitly. We do not define `__lt__`: there is
no single natural order for points. `sorted(..., key=...)` makes a chosen ordering explicit.
Because these points are mutable and compare by value, they are not hashable dictionary keys.


In [ ]:
from math import hypot

class Point:
    """A mutable two-dimensional point with addition and coordinate iteration."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def rotate_90_ccw(self):
        """Rotate this point 90 degrees counterclockwise around the origin."""
        self.x, self.y = -self.y, self.x

    def distance_to(self, other):
        if not isinstance(other, Point):
            raise TypeError("other must be a Point")
        return hypot(self.x - other.x, self.y - other.y)

    def __add__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return Point(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __iter__(self):
        return iter((self.x, self.y))

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"

    def __str__(self):
        return f"({self.x}, {self.y})"

point = Point(3, 5)
other = Point(9, -2)
point.rotate_90_ccw()
print(point, repr(point), repr(point + other))
print(sum([point, other], start=Point()))
print(tuple(point), point == Point(-5, 3))
print(sorted([other, point], key=lambda value: (value.x, value.y)))
try:
    point + 3
except TypeError:
    print("A Point can only be added to another Point.")


## 17. Syntax errors and runtime exceptions

A syntax error prevents a piece of source code from being parsed. A runtime exception occurs
while valid code is executing. A traceback shows the call path, the exception type, and its
message; the final line is a good place to identify the immediate problem.

The invalid syntax below is inside a string passed to `compile`, allowing us to catch the
`SyntaxError` without breaking the notebook. Each other demonstration catches exactly the
error it expects. In real code, a `TypeError` often means the operation received the wrong
kind of object; a `ValueError` often means the type was acceptable but its value was not.


In [ ]:
try:
    compile('while True print("Hello")', "<syntax demo>", "exec")
except SyntaxError as error:
    print(type(error).__name__, error.msg)

examples = [
    (ZeroDivisionError, lambda: 10 / 0),
    (NameError, lambda: missing_example_name),
    (TypeError, lambda: "2" + 2),
    (ValueError, lambda: int("two")),
    (IndexError, lambda: [10][9]),
    (KeyError, lambda: {"name": "Ada"}["age"]),
    (AttributeError, lambda: object().missing),
]
for expected_type, operation in examples:
    try:
        operation()
    except expected_type as error:
        print(type(error).__name__, str(error))


## 18. Exceptions are objects in an inheritance hierarchy

An `except` clause matches the named class **and its subclasses**. `Exception` is the usual
base for application errors; `BaseException` also includes control signals such as
`KeyboardInterrupt`, `SystemExit`, and `GeneratorExit`. A bare `except:` catches those too,
which can make a program difficult to stop. Handle the specific errors you can recover from.

Useful families include `LookupError` (`IndexError`, `KeyError`), `ArithmeticError`
(`ZeroDivisionError`, `OverflowError`), `OSError` (file and operating system errors), and
`RuntimeError` (`NotImplementedError`, `RecursionError`). `UnboundLocalError` is a `NameError`;
`IndentationError` is a `SyntaxError`. `StopIteration` signals normal iterator exhaustion.
Put specific handlers before broad ones because only the first matching handler runs.


In [ ]:
print(issubclass(FileNotFoundError, OSError))
print(issubclass(UnboundLocalError, NameError))
print(issubclass(KeyboardInterrupt, Exception))
try:
    {}["missing"]
except KeyError as error:
    print("Specific handler:", error)
except LookupError:
    print("This broader handler is not reached.")


## 19. Handle a failure close to the operation that can fail

If a statement in `try` raises, the rest of that `try` block is skipped. Python runs the first
matching `except` handler. If there is none, the exception propagates to the caller. Catch
a tuple of types only when they need the same recovery action; `except (IndexError, KeyError)`
is an example. `except ValueError as error` binds the exception object for reporting.

Instead of blocking on interactive input, the next example tries a finite sequence of
candidate strings. Invalid text is recoverable; an unexpected `None` is a `TypeError` and
is allowed to propagate. Small `try` blocks avoid hiding unrelated mistakes.


In [ ]:
def parse_first_integer(candidates):
    """Return the first valid integer string; a finite input gives a finite retry.

    Invalid strings are skipped. Unsupported types such as None raise TypeError.
    """
    for candidate in candidates:
        try:
            return int(candidate)
        except ValueError:
            continue
    raise ValueError("no valid integer was supplied")

print(parse_first_integer(["hello", "3.5", "42"]))
try:
    parse_first_integer([None, "42"])
except TypeError:
    print("Unexpected input type reached the caller.")

for collection in ([10], {"name": "Ada"}):
    try:
        print(collection[2])
    except (IndexError, KeyError):
        print("No value at key/index 2.")


## 20. Raise meaningful exceptions and protect valid state

Use `raise ValueError("message")` to reject an invalid value. You may raise an exception
instance or an exception class; including a message usually makes diagnosis easier.
A custom exception subclasses `Exception` (directly or through an application base class)
and names a failure that a caller may want to handle separately.

The workshop validates a booking **before** changing `booked`. If the booking fails, the
existing state remains valid. Checks such as these belong in normal code; do not use
`assert` for required input validation, because Python's optimized mode can remove assertions.


In [ ]:
class CapacityError(Exception):
    """A booking would exceed the available capacity."""


class Workshop:
    """Keep bookings between zero and a fixed nonnegative integer capacity."""

    def __init__(self, capacity):
        if isinstance(capacity, bool) or not isinstance(capacity, int):
            raise TypeError("capacity must be an integer")
        if capacity < 0:
            raise ValueError("capacity must not be negative")
        self.capacity = capacity
        self.booked = 0

    def book(self, places=1):
        if isinstance(places, bool) or not isinstance(places, int):
            raise TypeError("places must be an integer")
        if places <= 0:
            raise ValueError("places must be positive")
        if self.booked + places > self.capacity:
            raise CapacityError("not enough places available")
        self.booked += places
        return self.capacity - self.booked

workshop = Workshop(capacity=3)
print("Remaining:", workshop.book(2))
try:
    workshop.book(2)
except CapacityError as error:
    print(type(error).__name__, str(error))
print("Still booked:", workshop.booked)


## 21. Re-raise or translate an exception deliberately

Inside an `except` handler, bare `raise` re-raises the current exception, preserving the
original failure. This is useful if a layer can record context or undo partial work but
cannot recover. An outer handler can still decide what to do.

When an implementation error should become a more useful public error, use
`raise NewError(...) from error` to preserve its cause. Avoid catching errors just to
pretend the operation succeeded. The outer handlers below keep the demonstration runnable.


In [ ]:
def read_required_integer(text):
    try:
        return int(text)
    except ValueError:
        print("Could not parse a required integer.")
        raise

try:
    read_required_integer("several")
except ValueError:
    print("The caller received the original error.")

class ConfigurationError(Exception):
    """A configuration value could not be interpreted."""

try:
    try:
        int("several")
    except ValueError as error:
        raise ConfigurationError("count must contain an integer") from error
except ConfigurationError as error:
    print(str(error), "| cause:", type(error.__cause__).__name__)


## 22. `else` separates success work from the protected operation

A `try` statement's `else` block runs if the `try` body finishes without an exception (and
without leaving through `return`, `break`, or `continue`). It is a good place for follow-up
work that should happen only after success. An exception raised in `else` is **not** caught
by the preceding `except` clauses. That keeps a handler intended for parsing from also
catching a later, unrelated error. An outer layer can handle that later error if appropriate.


In [ ]:
for text in ["12", "twelve"]:
    try:
        number = int(text)
    except ValueError:
        print(f"Cannot parse {text!r}.")
    else:
        print(f"Parsed {number}; doubled value is {number * 2}.")

try:
    try:
        value = int("10")
    except ValueError:
        print("Parse failed.")
    else:
        raise ValueError("Failure in a separate success step")
except ValueError as error:
    print("Outer handler:", error)


## 23. `finally` performs cleanup on the way out

`finally` runs as control leaves the `try` statement during normal execution, after a
handled exception, before an unhandled exception propagates, and on a `return`, `break`, or
`continue`. Use it for cleanup. A `return` or new exception in `finally` can replace the
pending result or exception, so avoid those patterns. Abrupt process termination is outside
this normal language guarantee.

The event list below lets you observe the order without accessing any external resource.
The outer handler catches the failure only after the cleanup event has been appended.


In [ ]:
events = []

def finish_job(fail=False):
    try:
        events.append("work")
        if fail:
            raise RuntimeError("job failed")
        return "done"
    finally:
        events.append("cleanup")

print(finish_job(), events)
events.clear()
try:
    finish_job(fail=True)
except RuntimeError:
    events.append("caller handled failure")
print(events)


## 24. EAFP: attempt the operation and handle the expected failure

**Easier to Ask for Forgiveness than Permission (EAFP)** means trying an operation and
catching its specific failure. **Look Before You Leap (LBYL)** means checking a condition
first. Both can be readable. EAFP is especially helpful for files: a file can disappear
between an existence check and an attempted open, so the operation still needs error handling.

Here `pop` is the operation we actually need. Catching only `IndexError` handles an empty
list without hiding another problem, such as receiving an object with no `pop` method.
The function intentionally mutates its list by removing its final element.


In [ ]:
def pop_or_default(items, default=None):
    """Remove the final item, returning a default only when the list is empty."""
    try:
        return items.pop()
    except IndexError:
        return default

pending = ["first", "second"]
print(pop_or_default(pending), pending)
print(pop_or_default([], default="nothing pending"))


## 25. Context managers: structured setup and cleanup

`with manager as resource:` calls `manager.__enter__()` and assigns **its return value**
to `resource`. After the body, Python calls `__exit__(exception_type, exception, traceback)`.
The arguments are `None` on normal exit. Returning a truthy value suppresses a body
exception; returning `False` or `None` lets it propagate. If `__enter__` itself fails,
`__exit__` is not called, so setup must handle its own partial failure.

For files, prefer `with open(path, encoding="utf-8") as stream:`; it closes the file on
normal and exceptional exit. `StringIO` provides the same idea using text held in memory.
The small custom context manager below shows the exact exit signature and deliberately
does not suppress errors. There is no database dependency in these examples.


In [ ]:
from io import StringIO

with StringIO("Python\nObjects\n") as stream:
    first_line = stream.readline().strip()
print(first_line, "closed:", stream.closed)

class TrackedResource:
    def __init__(self):
        self.closed = False

    def __enter__(self):
        return self

    def __exit__(self, exception_type, exception, traceback):
        self.closed = True
        return False

resource = TrackedResource()
try:
    with resource as active:
        print("Same resource:", active is resource)
        raise ValueError("demonstration failure")
except ValueError:
    print("Failure propagated; resource closed:", resource.closed)


## Practice

Try each exercise before opening the solution notebook. Keep examples that distinguish a
correct implementation from a plausible mistake: separate instances, empty collections,
unsupported types, and failed operations that must leave state unchanged. The starter cells
contain only comments, so unfinished exercises do not interrupt Run All.


### Exercise 1 · Independent task lists

Create `TaskList(owner, tasks=None)`. Copy any supplied iterable into an instance list.
Implement `add_task(text)` and a `__repr__` containing the owner and tasks. Create two lists,
add a task to one, and prove the other is unchanged. Also check that mutating a caller's
original list does not change the stored tasks. Do not use a mutable class attribute or
mutable default argument. What does `first.add_task.__self__` refer to?


In [ ]:
# TODO: Define TaskList, then demonstrate independent lists and a bound method.
# Suggested checks after implementation:
# first = TaskList("Ada")
# second = TaskList("Lin")
# first.add_task("read")
# assert second.tasks == []


### Exercise 2 · Point distance and safe addition

Build a `Point` with defaults `(0, 0)`, value equality, addition, rotation, and a
`distance_to(other)` method using `math.hypot`. Unsupported addition should return
`NotImplemented`; unsupported distance arguments should raise `TypeError`. Verify distance
from `(0, 0)` to `(3, 4)`, four successive rotations, addition without input mutation,
`sum(points, start=Point())`, and an empty sum with the same start value.


In [ ]:
# TODO: Build or adapt Point and add assertions for the required edge cases.
# Suggested check: assert Point().distance_to(Point(3, 4)) == 5.0


### Exercise 3 · Extend behavior with inheritance

Create `Course(title)` with `describe()` returning its title. Create `OnlineCourse` with an
additional platform, using `super()` for initialization and description. Then implement a
diamond of classes `Root`, `Left(Root)`, `Right(Root)`, `Diamond(Left, Right)`: each of the
first three has a `labels()` method; `Left` and `Right` prepend their own label and delegate
with `super()`. Predict the result and verify it against `Diamond.mro()`.


In [ ]:
# TODO: Implement the single-inheritance and cooperative diamond examples.
# Explain in a comment why super() in Left continues to Right for a Diamond.


### Exercise 4 · A reusable container

Implement `ReadingList(titles=())` with `len`, membership, indexing/slicing, and iteration.
Copy the supplied titles. Verify negative indexing, an empty list, missing membership, and
two complete iterations. Check that an invalid index raises `IndexError`. Why should this
container return a new iterator from `__iter__` rather than return `self`?


In [ ]:
# TODO: Implement ReadingList and test its container protocol.
# Suggested checks: assert len(ReadingList()) == 0
# assert list(ReadingList(["Python"])) == ["Python"]


### Exercise 5 · A workshop with meaningful failures

Create `CapacityError(Exception)` and `Workshop(capacity)`. Capacity must be a nonnegative
integer, excluding `bool`. `book(places=1)` accepts only positive integers (also excluding
`bool`), raises `CapacityError` if capacity would be exceeded, and returns the remaining
places on success. Reject wrong types with `TypeError` and invalid values with `ValueError`.
Test full and zero-capacity workshops, and verify every rejected booking leaves `booked`
unchanged. Catch only the errors you expect in each test.


In [ ]:
# TODO: Define CapacityError and Workshop, then test success and rejected bookings.
# Suggested check: failed overbooking must not change the previous booked count.


### Exercise 6 · Parse text with cleanup

Write `read_integer(stream, events)` using a supplied `StringIO` stream as a context manager.
Read and convert its text inside a small `try` block. On `ValueError`, append `"invalid"`
to `events` and re-raise. In `else`, append `"parsed"` and return the integer. In `finally`,
append `"cleanup"`. Verify the event order and that the stream is closed for both valid
and invalid text. Also verify an unexpected error is allowed to reach the caller.


In [ ]:
# TODO: Import StringIO, implement read_integer, and test both exit paths.
# Expected success events: ["parsed", "cleanup"]
# Expected invalid-text events: ["invalid", "cleanup"]


## Check your understanding and continue

You should now be able to explain why `self` appears in a definition but not an instance
call, identify accidental shared state, follow `super()` through an MRO, and separate
expected recovery from cleanup. Special methods connect your classes to ordinary Python
operations; exceptions are a practical example of an inheritance hierarchy.

The next chapter, [06 · Functional programming](06_functional_programming.ipynb), develops
functions, iterators, generators, and decorators. For reference, consult Python's official
[classes tutorial](https://docs.python.org/3/tutorial/classes.html),
[data model](https://docs.python.org/3/reference/datamodel.html), and
[errors and exceptions tutorial](https://docs.python.org/3/tutorial/errors.html).

This chapter adapts the concepts in `5_oop.pdf`, pages 1–89, with current Python semantics
and fully runnable examples. The opening functional recap is expanded in Chapter 06.
